# Regresión de latitud y longitud con ResNet18 desde cero

ResNet18 con pesos aleatorios (`weights=None`) y **dos salidas lineales** en el
orden `[latitud / 90, longitud / 180]`. Todas las capas se entrenan. La función
objetivo es **MSE sobre las dos coordenadas normalizadas**, promediada sobre
coordenadas e imágenes. Para recuperar grados se multiplica por `[90, 180]`.

Conservamos el sample configurable, los splits separados por panorama, las
transformaciones deterministas sin aumentación y Adam con `lr=0.0001` del notebook
de clasificación. Este notebook guarda sus resultados en una carpeta diferente.

Orden: **1. Configuración y datos → 2. Entrenamiento y curvas de loss →
3. Métricas sobre validación**. La MSE en coordenadas no equivale a distancia
geográfica: por ejemplo, longitudes cercanas a −180° y +180° pueden estar próximas
en la Tierra y alejadas según esta loss. La distancia en km se usa para evaluar,
no para optimizar el modelo.

## 1. Configuración y datos

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from datasets import Dataset as HFDataset, Image as HFImage
from PIL import Image
from io import BytesIO
from tqdm.auto import tqdm
from IPython.display import display

# Rutas absolutas o relativas a la raíz del proyecto.
RUTA_SAMPLE = "datasets/samples/muestra_n-20000_umbral-json-1000_balanceado_paises-71_seed-42_val-20pct_por-panoid"
# None: nueva carpeta por ejecución. Para evaluar un modelo previo, indicar su carpeta.
RUTA_RESULTADOS = None
RUTA_REGISTRO = "src/notebooks/modeling/results.json"
ORDEN_COORDENADAS = ["latitude", "longitude"]
ESCALA_COORDENADAS = np.array([90.0, 180.0], dtype=np.float32)
RADIO_TIERRA_KM = 6371.0  # Distancias aproximadas sobre una esfera.
EPOCAS = 100
BATCH_SIZE = 4
ACUMULAR_BATCHES = 8  # Batch efectivo: 4 * 8 = 32 imágenes.
LEARNING_RATE = 0.0001
WEIGHT_DECAY = 5e-4
PACIENCIA = 5  # Early stopping según loss de validación.
TAMANO_IMAGEN = 224
NUM_WORKERS = 0  # Compatible con notebooks en Windows.
SEMILLA = 42
USAR_AMP = True  # Precisión mixta solo con CUDA.
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"

raiz = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "pyproject.toml").is_file()), None)
if raiz is None:
    raise FileNotFoundError("Ejecutá el notebook dentro del proyecto.")
def resolver_ruta(ruta):
    p = Path(ruta).expanduser()
    return (p if p.is_absolute() else raiz / p).resolve()

ruta_registro = resolver_ruta(RUTA_REGISTRO)
ruta_sample = resolver_ruta(RUTA_SAMPLE)
if not (ruta_sample / "COMPLETO").is_file():
    raise FileNotFoundError(f"Falta el marcador COMPLETO en {ruta_sample}.")
config_sample = json.loads((ruta_sample / "config.json").read_text(encoding="utf-8"))
archivos = {split: ruta_sample / config_sample["archivos_salida"][split]
            for split in ("train", "validation")}
for archivo in archivos.values():
    if not archivo.is_file():
        raise FileNotFoundError(archivo)
assert EPOCAS >= 1 and BATCH_SIZE >= 1 and ACUMULAR_BATCHES >= 1
assert PACIENCIA >= 1 and TAMANO_IMAGEN >= 32 and NUM_WORKERS >= 0
random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEMILLA)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
device = torch.device(DISPOSITIVO)
amp_activo = USAR_AMP and device.type == "cuda"
ruta_resultados = (resolver_ruta(RUTA_RESULTADOS) if RUTA_RESULTADOS else
                   raiz / "outputs" / "resnet18_regression" / datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
print(f"Sample: {ruta_sample}\nResultados: {ruta_resultados}\nDispositivo: {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(device))

Sample: D:\facuult\geo-proy\datasets\samples\muestra_n-20000_umbral-json-1000_balanceado_paises-71_seed-42_val-20pct_por-panoid
Resultados: d:\facuult\geo-proy\outputs\resnet18_regression\20261018_211622_016294
Dispositivo: cuda
NVIDIA GeForce GTX 1650


### Carga y controles de separación

La caché Arrow se comparte con el notebook de clasificación. Las imágenes se
decodifican al solicitarlas. Se verifica que no haya imágenes ni panoramas
compartidos entre splits y que todas las coordenadas sean finitas y estén dentro
de sus rangos: latitud en [−90, 90] y longitud en [−180, 180]. No se eliminan filas.
El país se conserva únicamente para agrupar las métricas por continente;
no es una entrada ni una etiqueta de entrenamiento.

In [3]:
datos = {
    split: HFDataset.from_parquet(str(archivo), cache_dir=str(raiz / ".cache/modeling/hf"))
        .cast_column("image", HFImage(decode=False))
    for split, archivo in archivos.items()
}

def metadatos(dataset):
    columnas = ["country_code", "panoid", "image_id", *ORDEN_COORDENADAS]
    tabla = dataset.select_columns(columnas).to_pandas()
    if tabla.isna().any().any():
        raise ValueError("Hay coordenadas, países o identificadores faltantes en el sample.")
    tabla["country_code"] = tabla["country_code"].str.strip().str.upper()
    if (tabla["country_code"] == "").any():
        raise ValueError("Hay códigos de país vacíos.")
    coords = tabla[ORDEN_COORDENADAS].to_numpy(dtype=np.float64)
    if not np.isfinite(coords).all() or (np.abs(coords) > ESCALA_COORDENADAS).any():
        raise ValueError("Hay coordenadas no finitas o fuera de los rangos geográficos.")
    return tabla

meta_train, meta_val = (metadatos(datos[s]) for s in ("train", "validation"))
if not len(meta_train) or not len(meta_val):
    raise ValueError("Ambos splits deben contener imágenes.")
for columna in ("panoid", "image_id"):
    if set(meta_train[columna]) & set(meta_val[columna]):
        raise ValueError(f"Fuga de datos: {columna} compartidos entre splits.")
paises = sorted(set(meta_train["country_code"]) | set(meta_val["country_code"]))
display(pd.DataFrame({
    "train": meta_train["country_code"].value_counts(),
    "validation": meta_val["country_code"].value_counts(),
}).fillna(0).astype(int).sort_index())
print(f"Train: {len(meta_train):,}; validación: {len(meta_val):,}")
display(meta_train[ORDEN_COORDENADAS].describe())

Generating train split: 16018 examples [00:08, 1937.16 examples/s]
Generating train split: 3982 examples [00:01, 2756.54 examples/s]


,train,validation
country_code,,
AR,226,56
AT,226,56
AU,226,56
BA,226,56
BD,226,56
...,...,...
UA,225,56
US,225,56
UY,225,56


Train: 16,018; validación: 3,982


,latitude,longitude
count,16018.000000,16018.000000
mean,25.840366,21.623534
std,29.610143,62.255488
min,-54.606129,-150.261645
25%,6.663079,-3.593589
50%,38.504188,18.017038
75%,49.024588,56.732809
max,71.012238,178.278816


### Convención de continentes

Agrupación explícita y editable de seis continentes: América del Norte incluye
Centroamérica y el Caribe. Rusia se asigna a Europa; Turquía y Kazajistán a Asia.
Se verifica la cobertura de todos los países del sample; un país nuevo requiere
agregar su código a esta tabla antes de entrenar.

In [4]:
GRUPOS_CONTINENTES = {
    "África": "BW GH KE NA NG SN ZA",
    "América del Norte": "CA CR MX US",
    "América del Sur": "AR BO BR CL CO EC PE PY UY",
    "Asia": "BD ID IN JP KH KR KZ LK MN MY NP OM PH TH TR TW VN",
    "Europa": "AT BA BE BG CH CZ DE DK EE ES FI FR GB GR HR HU IE IS IT LT LV NL NO PL PT RO RS RU SE SI SK UA",
    "Oceanía": "AU NZ",
}
pais_a_continente = {pais: continente
                     for continente, codigos in GRUPOS_CONTINENTES.items()
                     for pais in codigos.split()}
sin_continente = set(paises) - set(pais_a_continente)
if sin_continente:
    raise ValueError(f"Completá GRUPOS_CONTINENTES para: {sorted(sin_continente)}")
continentes = sorted({pais_a_continente[p] for p in paises})

In [5]:
# Escala fija [-1, 1]; no se utilizan pesos ni estadísticas aprendidas de otro dataset.
transform_train = transforms.Compose([
    transforms.Resize(TAMANO_IMAGEN),
    transforms.CenterCrop(TAMANO_IMAGEN),
    transforms.ToTensor(),
    transforms.Normalize((0.5,) * 3, (0.5,) * 3),
])
transform_val = transforms.Compose([
    transforms.Resize(TAMANO_IMAGEN),
    transforms.CenterCrop(TAMANO_IMAGEN),
    transforms.ToTensor(),
    transforms.Normalize((0.5,) * 3, (0.5,) * 3),
])

class ImagenesCoordenadas(Dataset):
    def __init__(self, datos, metadatos, transform):
        self.datos = datos
        self.objetivos = (metadatos[ORDEN_COORDENADAS].to_numpy(dtype=np.float32)
                          / ESCALA_COORDENADAS)
        self.transform = transform

    def __len__(self):
        return len(self.datos)

    def __getitem__(self, idx):
        contenido = self.datos[int(idx)]["image"]
        if contenido.get("bytes") is not None:
            fuente = BytesIO(contenido["bytes"])
        else:
            fuente = resolver_ruta(contenido["path"])
        with Image.open(fuente) as imagen:
            tensor = self.transform(imagen.convert("RGB"))
        return tensor, torch.from_numpy(self.objetivos[idx].copy())

dataset_train = ImagenesCoordenadas(datos["train"], meta_train, transform_train)
dataset_val = ImagenesCoordenadas(datos["validation"], meta_val, transform_val)
loader_train = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True,
                          generator=torch.Generator().manual_seed(SEMILLA),
                          num_workers=NUM_WORKERS, pin_memory=device.type == "cuda")
loader_val = DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=device.type == "cuda")

## 2. Fase de entrenamiento

ResNet18 desde cero, salida lineal de dimensión 2, MSE y Adam con los mismos
hiperparámetros del clasificador. Se conservan la acumulación de gradientes,
la precisión mixta en CUDA, el scheduler y el early stopping según loss de
validación. BatchNorm usa estadísticas del batch durante entrenamiento y las
estadísticas acumuladas durante validación.

Las curvas muestran MSE en coordenadas normalizadas, no error en grados ni km.
Se guarda el checkpoint de menor loss de validación y el historial por época.

In [6]:
modelo = models.resnet18(weights=None, num_classes=2).to(device)
criterio = nn.MSELoss()
optimizador = torch.optim.Adam(modelo.parameters(), lr=LEARNING_RATE,
                               weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizador, factor=0.1, patience=2)
scaler = torch.amp.GradScaler("cuda", enabled=amp_activo)
print(f"Parámetros entrenables: {sum(p.numel() for p in modelo.parameters() if p.requires_grad):,}")

def ejecutar_epoca(modelo, loader, entrenar):
    modelo.train(entrenar)
    total_loss, total = 0.0, 0
    if entrenar:
        optimizador.zero_grad(set_to_none=True)
    with torch.set_grad_enabled(entrenar):
        for paso, (imagenes, etiquetas) in enumerate(tqdm(loader, desc="Train" if entrenar else "Validación")):
            imagenes = imagenes.to(device, non_blocking=True)
            etiquetas = etiquetas.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, enabled=amp_activo):
                coordenadas = modelo(imagenes)
                loss = criterio(coordenadas.float(), etiquetas)
            n = etiquetas.size(0)
            if entrenar:
                # Ponderar por imágenes, incluso en el último grupo incompleto.
                inicio_grupo = (paso // ACUMULAR_BATCHES) * ACUMULAR_BATCHES
                n_grupo = min(ACUMULAR_BATCHES * loader.batch_size,
                              len(loader.dataset) - inicio_grupo * loader.batch_size)
                scaler.scale(loss * n / n_grupo).backward()
                if (paso + 1) % ACUMULAR_BATCHES == 0 or paso + 1 == len(loader):
                    scaler.step(optimizador)
                    scaler.update()
                    optimizador.zero_grad(set_to_none=True)
            total_loss += loss.detach().item() * n
            total += n
    return total_loss / total

Parámetros entrenables: 11,177,538


In [7]:
ruta_resultados.mkdir(parents=True, exist_ok=True)
ruta_checkpoint = ruta_resultados / "mejor_resnet18_regression.pt"
if ruta_checkpoint.exists():
    raise FileExistsError("Ya existe un modelo en esta carpeta. Usá otra RUTA_RESULTADOS para entrenar.")
config_entrenamiento = {
    "arquitectura": "resnet18", "tarea": "regresion_coordenadas", "weights": None, "ruta_sample": str(ruta_sample),
    "epocas_maximas": EPOCAS, "batch_size": BATCH_SIZE, "acumular_batches": ACUMULAR_BATCHES,
    "learning_rate": LEARNING_RATE, "optimizador": "Adam", "weight_decay": WEIGHT_DECAY,
    "paciencia": PACIENCIA, "tamano_imagen": TAMANO_IMAGEN, "semilla": SEMILLA,
    "amp": amp_activo, "pais_a_continente": pais_a_continente,
    "loss": "MSE", "orden_coordenadas": ORDEN_COORDENADAS,
    "escala_coordenadas": ESCALA_COORDENADAS.tolist(), "salida": "lineal",
    "preprocesamiento_train": repr(transform_train), "preprocesamiento_val": repr(transform_val),
    "radio_tierra_km": RADIO_TIERRA_KM,
    "filas_train": len(dataset_train), "filas_validation": len(dataset_val),
    "torch": str(torch.__version__), "dispositivo": str(device),
}
(ruta_resultados / "config_entrenamiento.json").write_text(
    json.dumps(config_entrenamiento, ensure_ascii=False, indent=2), encoding="utf-8")
historial = []
mejor_loss, sin_mejora = float("inf"), 0
for epoca in range(1, EPOCAS + 1):
    inicio = time.perf_counter()
    lr = optimizador.param_groups[0]["lr"]
    loss_train = ejecutar_epoca(modelo, loader_train, entrenar=True)
    loss_val = ejecutar_epoca(modelo, loader_val, entrenar=False)
    if not np.isfinite([loss_train, loss_val]).all():
        raise RuntimeError("Loss no finita. Revisá el learning rate y los datos.")
    historial.append({"epoca": epoca, "loss_train": loss_train, "loss_validation": loss_val,
                      "learning_rate": lr, "segundos": time.perf_counter() - inicio})
    pd.DataFrame(historial).to_csv(ruta_resultados / "historial.csv", index=False)
    if loss_val < mejor_loss:
        mejor_loss, sin_mejora = loss_val, 0
        torch.save({"model_state_dict": modelo.state_dict(), "epoca": epoca,
                    "loss_validation": loss_val, "config": config_entrenamiento}, ruta_checkpoint)
    else:
        sin_mejora += 1
    scheduler.step(loss_val)
    print(f"Época {epoca}/{EPOCAS} | train={loss_train:.4f} | val={loss_val:.4f} | lr={lr:.5g}")
    if sin_mejora >= PACIENCIA:
        print(f"Early stopping: {PACIENCIA} épocas sin mejora.")
        break
print(f"Mejor loss de validación: {mejor_loss:.4f}; checkpoint: {ruta_checkpoint}")

Train:  16%|█▌        | 643/4005 [01:23<07:18,  7.67it/s]


KeyboardInterrupt: 

In [ ]:
historia = pd.read_csv(ruta_resultados / "historial.csv")
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(historia["epoca"], historia["loss_train"], label="Train")
ax.plot(historia["epoca"], historia["loss_validation"], label="Validación")
mejor_epoca = int(historia.loc[historia["loss_validation"].idxmin(), "epoca"])
ax.axvline(mejor_epoca, color="gray", linestyle="--", label=f"Mejor época: {mejor_epoca}")
ax.set(xlabel="Época", ylabel="MSE media de coordenadas normalizadas", title="ResNet18 — regresión de coordenadas: loss")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(ruta_resultados / "curvas_loss.png", dpi=160)
plt.show()

## 3. Métricas sobre validación

Se recarga el mejor checkpoint y se evalúa **todo el split de validación** en
modo evaluación, sin gradientes. Para evaluar una corrida anterior sin entrenar,
indicar su carpeta en `RUTA_RESULTADOS`, ejecutar la sección 1 y luego esta sección.

Se reportan error geográfico medio y mediano en km, y MAE de latitud y longitud
en grados, globalmente y por continente real. La distancia se calcula sobre una
esfera mediante Haversine con radio de 6371 km; el MAE en grados utiliza la
diferencia directa de cada coordenada, igual que el objetivo de regresión.

Las dos salidas son lineales: no se agrega `tanh` ni se recortan las predicciones.
Se informa cuántas están fuera de los rangos geográficos convencionales. Haversine
interpreta los ángulos con su periodicidad trigonométrica, incluso en esos casos;
los grados originales se conservan para MAE y para el CSV. Esto permite detectar
salidas fuera de rango sin ocultarlas al evaluar.

Como validación también selecciona el checkpoint, estas métricas no reemplazan
una evaluación final sobre un conjunto de test independiente.

In [ ]:
ruta_checkpoint = ruta_resultados / "mejor_resnet18_regression.pt"
checkpoint = torch.load(ruta_checkpoint, map_location="cpu", weights_only=True)
config_guardada = checkpoint["config"]
if (config_guardada["arquitectura"] != "resnet18"
        or config_guardada.get("tarea") != "regresion_coordenadas"):
    raise ValueError("El checkpoint no corresponde a ResNet18 de regresión.")
if (Path(config_guardada["ruta_sample"]).resolve() != ruta_sample
        or config_guardada["tamano_imagen"] != TAMANO_IMAGEN
        or config_guardada["orden_coordenadas"] != ORDEN_COORDENADAS
        or config_guardada["escala_coordenadas"] != ESCALA_COORDENADAS.tolist()
        or config_guardada["salida"] != "lineal"
        or config_guardada["pais_a_continente"] != pais_a_continente
        or config_guardada["preprocesamiento_val"] != repr(transform_val)
        or config_guardada["radio_tierra_km"] != RADIO_TIERRA_KM):
    raise ValueError("La configuración de evaluación no coincide con el entrenamiento.")
if "optimizador" in globals():
    optimizador.state.clear()
if "modelo" not in globals():
    modelo = models.resnet18(weights=None, num_classes=2).to(device)
modelo.load_state_dict(checkpoint["model_state_dict"])
print(f"Evaluando época {checkpoint['epoca']} (MSE val={checkpoint['loss_validation']:.6f})")
info_mejor_modelo = {
    "epoca": int(checkpoint["epoca"]),
    "loss_validation": float(checkpoint["loss_validation"]),
}
del checkpoint
modelo.eval()
salidas = []
with torch.inference_mode():
    for imagenes, _ in tqdm(loader_val, desc="Métricas de validación"):
        with torch.autocast(device_type=device.type, enabled=amp_activo):
            coordenadas = modelo(imagenes.to(device, non_blocking=True))
        salidas.append(coordenadas.float().cpu().numpy())
pred_normalizadas = np.concatenate(salidas).astype(np.float64)
assert pred_normalizadas.shape == (len(dataset_val), 2)
assert len(dataset_val) == len(meta_val)
if not np.isfinite(pred_normalizadas).all():
    raise RuntimeError("El modelo produjo coordenadas no finitas.")
pred_grados = pred_normalizadas * ESCALA_COORDENADAS
reales_grados = meta_val[ORDEN_COORDENADAS].to_numpy(dtype=np.float64)
predicciones = meta_val[["image_id", "panoid", "country_code", *ORDEN_COORDENADAS]].copy()
predicciones = predicciones.rename(columns={"latitude": "latitud_real", "longitude": "longitud_real"})
predicciones["latitud_predicha"] = pred_grados[:, 0]
predicciones["longitud_predicha"] = pred_grados[:, 1]
predicciones["continente_real"] = predicciones["country_code"].map(pais_a_continente)
predicciones["error_abs_latitud_grados"] = np.abs(reales_grados[:, 0] - pred_grados[:, 0])
predicciones["error_abs_longitud_grados"] = np.abs(reales_grados[:, 1] - pred_grados[:, 1])
predicciones["fuera_de_rango"] = (np.abs(pred_grados) > ESCALA_COORDENADAS).any(axis=1)

In [ ]:
def distancia_haversine_km(reales, predichas):
    """Arrays (..., 2) en grados, orden latitud/longitud; devuelve km."""
    reales_rad = np.deg2rad(np.asarray(reales, dtype=np.float64))
    pred_rad = np.deg2rad(np.asarray(predichas, dtype=np.float64))
    lat_real, lon_real = reales_rad[..., 0], reales_rad[..., 1]
    lat_pred, lon_pred = pred_rad[..., 0], pred_rad[..., 1]
    a = (np.sin((lat_pred - lat_real) / 2) ** 2
         + np.cos(lat_real) * np.cos(lat_pred) * np.sin((lon_pred - lon_real) / 2) ** 2)
    # Protección frente al redondeo numérico, sin recortar las coordenadas.
    a = np.clip(a, 0.0, 1.0)
    return 2 * RADIO_TIERRA_KM * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

predicciones["error_km"] = distancia_haversine_km(reales_grados, pred_grados)
resumen_metricas = {
    "n_validation": len(predicciones),
    "mse_normalizada": float(np.mean((reales_grados / ESCALA_COORDENADAS - pred_normalizadas) ** 2)),
    "error_medio_km": float(predicciones["error_km"].mean()),
    "error_mediano_km": float(predicciones["error_km"].median()),
    "mae_latitud_grados": float(predicciones["error_abs_latitud_grados"].mean()),
    "mae_longitud_grados": float(predicciones["error_abs_longitud_grados"].mean()),
    "predicciones_fuera_de_rango": int(predicciones["fuera_de_rango"].sum()),
}
display(pd.Series(resumen_metricas, name="Validación").to_frame())
por_continente = predicciones.groupby("continente_real").agg(
    n=("error_km", "size"), error_medio_km=("error_km", "mean"),
    error_mediano_km=("error_km", "median"),
    mae_latitud_grados=("error_abs_latitud_grados", "mean"),
    mae_longitud_grados=("error_abs_longitud_grados", "mean"),
    predicciones_fuera_de_rango=("fuera_de_rango", "sum"))
display(por_continente)
predicciones.to_csv(ruta_resultados / "predicciones_validation.csv", index=False)
por_continente.to_csv(ruta_resultados / "metricas_por_continente.csv")
(ruta_resultados / "metricas_validation.json").write_text(
    json.dumps(resumen_metricas, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")

### Registro de modelos en results.json

Al terminar la evaluacion, se agrega o reemplaza `regressor_<cantidad_datos>` en
`src/notebooks/modeling/results.json`. La cantidad incluye train y validacion;
por ejemplo, `regressor_20000`. Repetir esa cantidad reemplaza su entrada, incluso
si cambia la seed, y conserva los resultados de otras cantidades o modelos.

Cada entrada incluye la configuracion y seed del checkpoint seleccionado por
menor MSE de validacion, la configuracion de la muestra, los conteos por split,
la mejor epoca, el historial y las metricas globales y por continente. Los pesos
siguen guardados en el checkpoint `.pt`, cuya ruta se incluye en el registro.


In [ ]:
def guardar_resultado_regresor(ruta_json, clave, informacion):
    """Agrega o reemplaza una corrida y conserva las otras claves del registro."""
    contenido = ruta_json.read_text(encoding="utf-8-sig") if ruta_json.exists() else ""
    resultados = json.loads(contenido) if contenido.strip() else {}
    if not isinstance(resultados, dict):
        raise ValueError("El registro de resultados debe ser un objeto JSON.")
    resultados[clave] = informacion
    contenido_nuevo = json.dumps(resultados, ensure_ascii=False, indent=2, allow_nan=False)
    ruta_json.parent.mkdir(parents=True, exist_ok=True)
    ruta_temporal = ruta_json.with_suffix(ruta_json.suffix + ".tmp")
    try:
        ruta_temporal.write_text(contenido_nuevo + "\n", encoding="utf-8")
        ruta_temporal.replace(ruta_json)
    finally:
        ruta_temporal.unlink(missing_ok=True)


# Usar la configuracion del checkpoint evaluado, incluso al evaluar una corrida previa.
cantidad_datos = int(config_guardada["filas_train"] + config_guardada["filas_validation"])
clave_registro = f"regressor_{cantidad_datos}"
historial_registrado = pd.read_csv(ruta_resultados / "historial.csv").to_dict(orient="records")
resultado_regresor = {
    "modelo": config_guardada["arquitectura"],
    "cantidad_datos": cantidad_datos,
    "seed": int(config_guardada["semilla"]),
    "filas_train": int(config_guardada["filas_train"]),
    "filas_validation": int(config_guardada["filas_validation"]),
    "mejor_modelo": info_mejor_modelo,
    "criterio_seleccion": "menor loss MSE de validacion",
    "configuracion_entrenamiento": config_guardada,
    "configuracion_muestra": config_sample,
    "epocas_ejecutadas": len(historial_registrado),
    "historial": historial_registrado,
    "metricas_validacion": resumen_metricas,
    "metricas_por_continente": por_continente.reset_index().to_dict(orient="records"),
    "parametros_totales": sum(p.numel() for p in modelo.parameters()),
    "parametros_entrenables": sum(p.numel() for p in modelo.parameters() if p.requires_grad),
    "evaluacion": {"dispositivo": str(device), "amp": amp_activo},
    "archivos": {
        "checkpoint": str(ruta_checkpoint),
        "resultados": str(ruta_resultados),
        "sample": str(ruta_sample),
    },
    "fecha_actualizacion": datetime.now().astimezone().isoformat(timespec="seconds"),
}
guardar_resultado_regresor(ruta_registro, clave_registro, resultado_regresor)
print(f"Registro guardado: {clave_registro} en {ruta_registro}")


### Gráficos de validación

Comparación de coordenadas reales y predichas (la diagonal indica predicción
perfecta) y distribución de distancias de error. Las figuras, predicciones,
métricas y curvas de loss se guardan en la carpeta de la corrida.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for i, (ax, nombre) in enumerate(zip(axes, ["Latitud", "Longitud"])):
    ax.scatter(reales_grados[:, i], pred_grados[:, i], s=8, alpha=0.4)
    minimo = min(reales_grados[:, i].min(), pred_grados[:, i].min())
    maximo = max(reales_grados[:, i].max(), pred_grados[:, i].max())
    ax.plot([minimo, maximo], [minimo, maximo], "--", color="gray")
    ax.set(xlabel=f"{nombre} real (grados)", ylabel=f"{nombre} predicha (grados)", title=nombre)
    ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(ruta_resultados / "coordenadas_reales_vs_predichas.png", dpi=160)
plt.show()

errores = np.sort(predicciones["error_km"].to_numpy())
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(errores, bins=40)
axes[0].set(xlabel="Error geográfico (km)", ylabel="Imágenes", title="Distribución del error")
axes[1].step(errores, np.arange(1, len(errores) + 1) / len(errores), where="post")
axes[1].set(xlabel="Error geográfico (km)", ylabel="Fracción acumulada", title="Error acumulado", ylim=(0, 1))
for ax in axes:
    ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(ruta_resultados / "distribucion_error_km.png", dpi=160)
plt.show()
print(f"Resultados guardados en {ruta_resultados}")